# Final model package (only `final1-fold0`)

Replaces `report_bundle/`, `report_bundle.zip` and `best_model/` on Drive with one folder, `dl_lab2/final_model/`, that holds files of the final model only.
`dl_lab2/results/` (all runs) is not touched.

In [1]:
import json, shutil
from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/dl_lab2")
if not ROOT.exists():
    from google.colab import drive
    drive.mount("/content/drive")
RUN = ROOT / "results" / "final1-fold0"
LABELS = ["Cat", "Dog", "Moris", "Motya", "Biatrix"]
OUT = ROOT / "final_model"
assert (RUN / "summary.json").exists() and (RUN / "best.ckpt").exists()

for old in ("report_bundle", "best_model"):
    shutil.rmtree(ROOT / old, ignore_errors=True)
(ROOT / "report_bundle.zip").unlink(missing_ok=True)
shutil.rmtree(OUT, ignore_errors=True)
OUT.mkdir()
s = json.loads((RUN / "summary.json").read_text())

In [2]:
val = pd.read_csv(RUN / "val_probs.csv", index_col=0)
test = pd.read_csv(RUN / "test_probs.csv", index_col=0)
assert list(test.columns) == LABELS and len(test) == 18673
thr = pd.Series({l: s[f"threshold_{l}"] for l in LABELS})

# 1. probabilities and submissions
val.rename_axis("image_id").to_csv(OUT / "val_probs.csv")
test.rename_axis("image_id").to_csv(OUT / "test_probs.csv")
(test >= 0.5).astype(int).rename_axis("image_id").reset_index().to_csv(OUT / "submission_05.csv", index=False)
(test >= thr).astype(int).rename_axis("image_id").reset_index().to_csv(OUT / "submission_tuned.csv", index=False)

# 2. metrics: one row overall, one row per class
overall = {k: s[k] for k in ("ap_macro", "f1_macro_05", "f1_macro_tuned", "exact_match_05")}
overall |= {"best_epoch": s["best_epoch"], "epochs_run": s["epochs_run"], "num_parameters": s["num_parameters"],
            "sec_per_epoch": s["sec_per_epoch"], "train_minutes": round(s["train_seconds"] / 60, 1)}
pd.DataFrame([overall]).to_csv(OUT / "metrics_overall.csv", index=False)
per_class = pd.DataFrame({
    "label": LABELS,
    "ap": [s[f"ap_{l}"] for l in LABELS],
    "f1_at_0.5": [s[f"f1_{l}"] for l in LABELS],
    "tuned_threshold": [s[f"threshold_{l}"] for l in LABELS],
    "test_positives_at_0.5": [(test[l] >= 0.5).sum() for l in LABELS],
})
per_class.to_csv(OUT / "metrics_per_class.csv", index=False)

# 3. training curves, one row per epoch (the per-step gradient norms are dropped)
long = pd.read_csv(RUN / "metrics.csv")
keep = [k for k in long.key.unique() if not k.endswith("_step") and not k.startswith("final_")]
wide = long[long.key.isin(keep)].pivot_table(index="step", columns="key", values="value").reset_index()
wide.to_csv(OUT / "metrics_per_epoch.csv", index=False)

# 4. config, plot, weights
(OUT / "config.json").write_text(json.dumps(s["config"], indent=2))
shutil.copy2(RUN / "curves.png", OUT / "curves.png")
shutil.copy2(RUN / "best.ckpt", OUT / "best.ckpt")
display(pd.DataFrame([overall]).T, per_class, wide.tail(3).T)
print(sorted(p.name for p in OUT.iterdir()))

,0
ap_macro,7.835738e-01
f1_macro_05,5.842999e-01
f1_macro_tuned,7.346420e-01
exact_match_05,8.038336e-01
best_epoch,2.500000e+01
epochs_run,3.000000e+01
num_parameters,1.227685e+06
sec_per_epoch,2.420000e+01
train_minutes,1.210000e+01


,label,ap,f1_at_0.5,tuned_threshold,test_positives_at_0.5
0,Cat,0.938472,0.861328,0.45,7814
1,Dog,0.913611,0.835329,0.50,7691
2,Moris,0.866088,0.468085,0.95,421
3,Motya,0.682906,0.432432,0.85,241
4,Biatrix,0.516793,0.324324,0.80,300


,283,284,285
key,,,
step,4559.0,4579.0,4589.000000
epoch,29.0,29.0,29.000000
grad_norm/classifier_epoch,NaN,NaN,0.226431
grad_norm/stage1_epoch,NaN,NaN,0.940078
grad_norm/stage2_epoch,NaN,NaN,0.760953
grad_norm/stage3_epoch,NaN,NaN,0.468793
grad_norm/stage4_epoch,NaN,NaN,0.196134
grad_norm/stem_epoch,NaN,NaN,1.828794
lr-AdamW,NaN,NaN,NaN


['best.ckpt', 'config.json', 'curves.png', 'metrics_overall.csv', 'metrics_per_class.csv', 'metrics_per_epoch.csv', 'submission_05.csv', 'submission_tuned.csv', 'test_probs.csv', 'val_probs.csv']


In [3]:
# the normalisation the weights were trained with (training part of fold 0), needed to feed new images
cache = Path("/content/CNN_for_multilabel_classification/data/cache/stats_224_fold0.json")
stats = json.loads(cache.read_text()) if cache.exists() else None
(OUT / "normalization_fold0.json").write_text(json.dumps(stats, indent=2))
print(stats)

{'mean': [0.4889, 0.4623, 0.42], 'std': [0.265, 0.2573, 0.2631]}


In [4]:
wide = pd.read_csv(OUT / "metrics_per_epoch.csv")
per_epoch = wide.groupby("epoch").first().reset_index()
per_epoch.to_csv(OUT / "metrics_per_epoch.csv", index=False)
print(per_epoch.shape)

(30, 24)
